# Demo 1 · Energy: what the board's power reading shows

Instructor demo. Runs three configurations on one 48-chip board, an empty board, a large silent network and the same network with every neuron spiking, and plots the board power during each run. Start section 2 in the break: the three runs take about 5 minutes together. The results and the figure are saved under the py-spinnaker2 version (`output/py-spinnaker2-<version>/`), so a figure from an earlier run can be shown if the chip is not available (section 4).

The message: the board needs a large baseline just for being on; a small network does not move the reading; spiking activity is what adds energy on top.

In [ ]:
import importlib.metadata
import shutil
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

N_CORES = 1000  # cores (PEs) of the board to load, out of 7,104
NEURONS_PER_CORE = 100
STEPS = 20000  # 20 s at 1 ms per step
MODES = ["empty", "silent", "active"]
COLORS = {"empty": "0.55", "silent": "#1f5fa8", "active": "#c0392b"}  # the same colours as in the diagram of section 1
LABELS = {"empty": "empty board (1 neuron)", "silent": f"{N_CORES:,} cores, silent", "active": f"{N_CORES:,} cores, every neuron spikes"}
VERSION = importlib.metadata.version("spinnaker2")  # note: the v0.8.2 install reports 0.8.0
OUTPUT = Path("output") / f"py-spinnaker2-{VERSION}"  # live runs and the figure, one folder per version
MEASURED = Path("../../05_measurements/01_power_baseline/data")  # fallback: the measurement of 2026-10-19 (v0.8.2)
print("py-spinnaker2", VERSION, "| results go to", OUTPUT)

## 1. The network and the three options

![The network on one core and the three options](figures/energy_network.png)

*The network on one core (left) and the three options, each on one 48-chip board (right). Diagram, generated by `figures/make_energy_network.py`.*

| Option | What runs | What it tests |
|---|---|---|
| **Empty board** | 1 `lif` neuron on 1 core, no input | the baseline: the board just being on |
| **1,000 cores, silent** | 1,000 cores, each with 100 `lif` neurons connected all-to-all (10,000 synapses per core), no input | loaded cores that tick every millisecond but never spike |
| **1,000 cores, active** | the same network with a constant input above the threshold | every neuron spikes in every step: 10,000 synaptic events per core and step, 10 billion per second on the board |

Silent and active differ only in the spikes, so the difference between them is what the spiking costs.

In [ ]:
def build(mode):
    """The network of one option: "empty", "silent" or "active"."""
    from spinnaker2 import snn

    params = {"threshold": 1.0, "alpha_decay": 0.9, "i_offset": 2.0 if mode == "active" else 0.0}  # i_offset 2 > threshold 1: spikes in every step
    net = snn.Network(f"energy_{mode}")
    if mode == "empty":
        net.add(snn.Population(size=1, neuron_model="lif", params=params, name="p0", record=[]))
        return net
    pre, post = np.meshgrid(np.arange(NEURONS_PER_CORE), np.arange(NEURONS_PER_CORE), indexing="ij")
    conns = np.stack([pre.ravel(), post.ravel(), np.ones(pre.size), np.zeros(pre.size)], axis=1).astype(int)  # all-to-all, weight 1, delay 0
    for k in range(N_CORES):  # one population per core
        pop = snn.Population(size=NEURONS_PER_CORE, neuron_model="lif", params=params, name=f"p{k}", record=[])
        net.add(pop, snn.Projection(pre=pop, post=pop, connections=conns))
    return net


def run(mode):
    """One run with power recording; stores the power trace as <OUTPUT>/<mode>.parquet."""
    from spinnaker2 import hardware
    from spinnaker2.tools import diagnostic_folder_utils as dfu

    dfu.set_level("persistent")
    hardware.SpiNNcloud48NodeBoard().run(build(mode), STEPS, record_power=True)
    OUTPUT.mkdir(parents=True, exist_ok=True)
    shutil.copy(dfu.get_path() / "power.parquet", OUTPUT / f"{mode}.parquet")
    print(f"{mode}: done")

## 2. Run on the chip (start in the break, about 5 minutes)

In [ ]:
for mode in MODES:
    run(mode)

## 3. Board power during the simulation

py-spinnaker2 reads the board's power from the frame's controller about every 0.1 s. The controller reports only multiples of 3 W and updates its value about once per second, so the curves move in steps. The dashed line is the average after the first 2 seconds, the dotted grey line the empty board (the reading lags up to a second behind the start). If a live run of this version is missing, its row shows the measurement of 2026-10-19 instead, and the title says so.

In [ ]:
def load(mode):
    """Live run of this version if available, otherwise the measurement of 2026-10-19."""
    live = OUTPUT / f"{mode}.parquet"
    if live.exists():
        return pd.read_parquet(live), f"live run, py-spinnaker2 {VERSION}"
    name = {"empty": "idle_1", "silent": f"silent_{N_CORES}", "active": f"active_{N_CORES}"}[mode]
    return pd.read_parquet(MEASURED / f"{name}.parquet"), "measured 2026-10-19, py-spinnaker2 v0.8.2"


def simulation_window(df):
    """Time since the start of the simulation and the board power, between the sim_start and sim_end markers."""
    marks = dict(zip(df.loc[df.marker.notna(), "marker"], df.loc[df.marker.notna(), "time_s"]))
    sim = df[(df.time_s > marks["sim_start"]) & (df.time_s < marks["sim_end"]) & df.total.notna()]
    return sim.time_s - marks["sim_start"], sim.total


fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True, sharey=True)  # one row per option, taller than the usual (10, 5)
all_live = True
t0, p0 = simulation_window(load("empty")[0])
baseline = p0[t0 > 2].mean()  # the empty board, drawn as a dotted reference in the other two rows
for ax, mode in zip(axes, MODES):
    df, source = load(mode)
    all_live &= source.startswith("live")
    t, power = simulation_window(df)
    average = power[t > 2].mean()
    ax.step(t, power, where="post", color=COLORS[mode], lw=2.5)
    ax.axhline(average, color=COLORS[mode], ls="--", lw=1.2)
    if mode != "empty":
        ax.axhline(baseline, color=COLORS["empty"], ls=":", lw=1.5)
        ax.text(0.2, baseline - 0.5, "empty board", color=COLORS["empty"], va="top", fontsize=9)
    ax.text(20.3, average, f"{average:.1f} W", color=COLORS[mode], va="center", fontsize=11, weight="bold")
    ax.set_title(f"{LABELS[mode]}  ({source})", loc="left", color=COLORS[mode], fontsize=10)
    ax.grid(axis="y", color="0.9")
ax.set(xlim=(0, 20), ylim=(55, 70), yticks=[57, 60, 63, 66, 69])  # the 3 W levels the controller reports
axes[-1].set_xlabel("time since start of the simulation (s)")
axes[1].set_ylabel("board power (W, 3 W steps)")
fig.suptitle(f"Board power of one 48-chip board, {date.today()}, py-spinnaker2 {VERSION}", fontsize=11)
plt.tight_layout()
if all_live:  # keep a copy of the live result for this version, for section 4
    fig.savefig(OUTPUT / f"energy_demo_py-spinnaker2-{VERSION}.png", dpi=150)
plt.show()

**Reading the plot:**

- **Empty board:** a flat line. This is the baseline the board needs just for being on, whatever runs on it; a workshop network of a few cores reads the same.
- **Silent, the ups and downs:** the true power of the silent network lies between two of the controller's 3 W levels, just above the empty board. The controller can only report one level or the other, so the reading flips between them about once per second; the share of time on the upper level shows roughly how far above the lower level the true value is. In the run of 2026-11-17 the reading spent about a third of the time on 60 W and averaged 58 W: the 1,000 ticking cores add only about 1 W, because they sleep most of each millisecond. Treat this as a rough estimate (why the steps are 3 W, and whether the flipping is a fair average, is not known).
- **Active:** a steady level well above the other two, after a short lag at the start. The difference to the silent run is what the spikes cost.
- The numbers differ by a level or two between runs and days (active 66 W on 2026-10-19, 63 W on 2026-11-09 and 2026-11-17); the order empty < silent < active held in every run until the morning of 2026-11-18; section 5 shows a run where it did not.

## 4. If the chip is not available

The figures saved by earlier live runs, one per py-spinnaker2 version.

In [ ]:
for png in sorted(Path("output").glob("py-spinnaker2-*/energy_demo_*.png")):
    print(png)
    display(Image(filename=png))

## 5. Two runs on the same day: why the comparison needs one board

The same notebook, run twice on 2026-11-18, once with py-spinnaker2 v0.8.2 (the environment reports itself as 0.8.0) and once with v0.8.5:

![Energy demo, 2026-11-18, py-spinnaker2 v0.8.2](figures/energy_demo_2026-11-18_v0.8.2.png)

*Run A, 08:32 to 08:35, v0.8.2: empty board 57 W, silent 57.2 W, active 60.5 W. The expected order.*

![Energy demo, 2026-11-18, py-spinnaker2 v0.8.5](figures/energy_demo_2026-11-18_v0.8.5.png)

*Run B, 10:42 to 10:46, v0.8.5: empty board 60 W, silent 57 W, active 60 W. The empty board reads higher than the silent network.*

- **Run B cannot be right as a comparison:** a board with one idle neuron does not draw more than the same board with 1,000 loaded cores.
- **The likely reason is the board, not the version:** every run takes the first free board, and the frame has six. The three runs of B probably landed on different boards whose baselines differ by a 3 W step. The log does not say which board a run used, so this is not verified. The chip code is the same in v0.8.2 and v0.8.5, so the version is an unlikely cause.
- **The 3 W steps make it worse:** a difference of one step is all the demo can show, so a board with a slightly higher baseline swaps the order.
- **What this teaches:** an energy number needs a baseline measured on the same hardware, in the same conditions, and more than one run. Comparing readings from different boards, or single runs, says little. The same holds for published comparisons with GPUs (`01_knowledge_base/02_hardware_energy.md`, section 8).

**Proposed fix (not applied here):** run all three options on the same board. py-spinnaker2 0.8.5 can choose one: `hardware.SpiNNcloud48NodeBoard(board_id=BOARD_ID)`. In the code of section 1, add a setting `BOARD_ID = 0` and pass it in `run()`; then repeat each option two or three times and compare the averages. Run it when no one else needs that board (a run waits while its board is busy).

## 6. What it shows

- The board needs a large baseline just for being on. Small networks are invisible in it: the 3 W steps cannot show a network of a few cores.
- Cores that only tick cost little: they sleep between the timer ticks.
- Spiking activity is what adds energy: the same network with every neuron spiking draws clearly more. That is the principle behind neuromorphic computing, and the reason sparse activity matters.
- The full measurement with up to 7,000 cores: `05_measurements/01_power_baseline/README.md`.